In [1]:
import pandas as pd
import numpy as np
import torch
import os

### Load all data

In [6]:
def load_go_df(seed):
    df_file_name = '../../03_results/reports/sc_dgd_sae_go_analysis_all.csv' if seed == 0 else f'../../03_results/reports/sc_dgd_sae_go_analysis_rs{seed}_all.csv'
    sub_df_stem = '../../03_results/reports/sc_dgd_sae_go_analysis_'
    sub_df_end = '.csv' if seed == 0 else f'_rs{seed}.csv'
    sub_df_names = ['small_2_biological_process', 'large_2_biological_process', 'small_2_molecular_function', 'large_2_molecular_function']
    if not os.path.exists(df_file_name):
        go_df = []
        for sub_df_name in sub_df_names:
            sub_df = pd.read_csv(f'{sub_df_stem}{sub_df_name}{sub_df_end}')
            sub_df['go type'] = 'biological process' if 'biological' in sub_df_name else 'molecular function'
            go_df.append(sub_df)
        go_df = pd.concat(go_df, axis=0, ignore_index=True)
    else:
        go_df = pd.read_csv(df_file_name)
    go_df['go_id'] = go_df['go_id'].astype('category')
    go_df['feature'] = go_df['feature'].astype('category')
    return go_df

def load_activations(seed):
    file_name = '../../03_results/reports/sae_model_10000_l1-1e-3_lr-1e-4_500epochs_activations.pt' if seed == 0 else f'../../03_results/reports/sae_model_10000_l1-1e-3_lr-1e-4_500epochs_seed{seed}_activations.pt'
    if not os.path.exists(file_name):
        raise FileNotFoundError(f"Activation file not found: {file_name}")
    activations = torch.load(file_name, weights_only=False)
    return activations

In [8]:
# multiDGD GO terms
# look at the dataframe with GO term analysis

print("Loading GO result dataframes")
go_df_dgd1 = load_go_df(0)
go_df_dgd2 = load_go_df(42)
go_df_dgd3 = load_go_df(9307)

print("Loading activations")
activations_dgd1 = load_activations(0)
activations_dgd2 = load_activations(42)
activations_dgd3 = load_activations(9307)

Loading GO result dataframes
Loading activations


### Compare the binary feature matrices

In [9]:
from goatools.obo_parser import GODag
obodag = GODag("../../01_data/go-basic.obo")

../../01_data/go-basic.obo: fmt(1.2) rel(2024-10-27) 44,017 Terms


In [10]:
from tqdm import tqdm
def create_go_feature_matrix(go_df, activations):
    go_feature_matrix = torch.zeros((len(go_df['go_name'].unique()), activations.shape[1]))
    for i, go_id in tqdm(enumerate(go_df['go_id'].unique())):
        for feat in go_df[go_df['go_id'] == go_id]['feature']:
            go_feature_matrix[i,feat] = 1
    mtrx_go_ids = go_df['go_id'].unique()
    mtrx_go_names = [obodag[x].name for x in mtrx_go_ids]
    mtrx_feature_ids = torch.where(go_feature_matrix.sum(dim=0) > 0)[0]
    go_feature_matrix = go_feature_matrix[:,torch.where(go_feature_matrix.sum(dim=0) > 0)[0]]
    return go_feature_matrix, mtrx_go_ids, mtrx_go_names, mtrx_feature_ids

In [12]:
go_feature_matrix_1, mtrx_go_ids_1, mtrx_go_names_1, mtrx_feature_ids_1 = create_go_feature_matrix(go_df_dgd1, activations_dgd1)
go_feature_matrix_2, mtrx_go_ids_2, mtrx_go_names_2, mtrx_feature_ids_2 = create_go_feature_matrix(go_df_dgd2, activations_dgd2)
go_feature_matrix_3, mtrx_go_ids_3, mtrx_go_names_3, mtrx_feature_ids_3 = create_go_feature_matrix(go_df_dgd3, activations_dgd3)

0it [00:00, ?it/s]

2499it [00:03, 652.18it/s] 
2310it [00:01, 1715.29it/s]
1966it [00:00, 3000.64it/s]


In [21]:
print("Number of unique GO terms per SAE:")
print(f"SAE1: {len(mtrx_go_ids_1)}")
print(f"SAE2: {len(mtrx_go_ids_2)}")
print(f"SAE3: {len(mtrx_go_ids_3)}")

Number of unique GO terms per SAE:
SAE1: 2499
SAE2: 2310
SAE3: 1966


In [13]:
shared_go_terms12 = sorted(set(mtrx_go_ids_1).intersection(set(mtrx_go_ids_2)))
shared_go_terms13 = sorted(set(mtrx_go_ids_1).intersection(set(mtrx_go_ids_3)))
shared_go_terms23 = sorted(set(mtrx_go_ids_2).intersection(set(mtrx_go_ids_3)))
shared_go_ids12 = [x for x in mtrx_go_ids_1 if obodag[x].name in shared_go_terms12]
shared_go_ids13 = [x for x in mtrx_go_ids_1 if obodag[x].name in shared_go_terms13]
shared_go_ids23 = [x for x in mtrx_go_ids_2 if obodag[x].name in shared_go_terms23]

In [14]:
# print the fractions of shared terms per pair
print("Fraction of shared GO terms between SAE1 and SAE2:", len(shared_go_terms12) / max(len(mtrx_go_ids_1), len(mtrx_go_ids_2)))
print("Fraction of shared GO terms between SAE1 and SAE3:", len(shared_go_terms13) / max(len(mtrx_go_ids_1), len(mtrx_go_ids_3)))
print("Fraction of shared GO terms between SAE2 and SAE3:", len(shared_go_terms23) / max(len(mtrx_go_ids_2), len(mtrx_go_ids_3)))

Fraction of shared GO terms between SAE1 and SAE2: 0.9235694277711084
Fraction of shared GO terms between SAE1 and SAE3: 0.7859143657462985
Fraction of shared GO terms between SAE2 and SAE3: 0.8510822510822511


In [20]:
print("Fraction of shared GO terms between DGD SAE1 and Geneformer SAE:", 2499 / (2499+97))

Fraction of shared GO terms between DGD SAE1 and Geneformer SAE: 0.9626348228043143


In [15]:
go_feature_matrix1_shared2 = go_feature_matrix_1[[np.where(mtrx_go_ids_1 == x)[0][0] for x in shared_go_ids12],:].T
go_feature_matrix1_shared3 = go_feature_matrix_1[[np.where(mtrx_go_ids_1 == x)[0][0] for x in shared_go_ids13],:].T
go_feature_matrix2_shared1 = go_feature_matrix_2[[np.where(mtrx_go_ids_2 == x)[0][0] for x in shared_go_ids12],:].T
go_feature_matrix2_shared3 = go_feature_matrix_2[[np.where(mtrx_go_ids_2 == x)[0][0] for x in shared_go_ids23],:].T
go_feature_matrix3_shared1 = go_feature_matrix_3[[np.where(mtrx_go_ids_3 == x)[0][0] for x in shared_go_ids13],:].T
go_feature_matrix3_shared2 = go_feature_matrix_3[[np.where(mtrx_go_ids_3 == x)[0][0] for x in shared_go_ids23],:].T

In [16]:
# compare with optimal bipartite matching
# since we have binary matrices, we can use the Jaccard index

from scipy.spatial.distance import jaccard
from scipy.optimize import linear_sum_assignment

def calculate_matrix_similarity(matrix_a: np.ndarray, matrix_b: np.ndarray) -> float:
    """
    Calculates the similarity between two binary matrices with shared columns
    but unsorted and unpaired rows.

    The method finds the optimal matching between rows of the two matrices
    using the Jaccard similarity and the Hungarian algorithm (linear_sum_assignment),
    and then computes the average similarity of these matched pairs.

    Args:
        matrix_a (np.ndarray): The first binary matrix with shape (n, k).
                               Rows are features/entities, columns are attributes.
        matrix_b (np.ndarray): The second binary matrix with shape (m, k).
                               Rows are features/entities, columns are attributes.
                               Must have the same number of columns (k) as matrix_a.

    Returns:
        float: A similarity score between 0.0 and 1.0.
               1.0 if both matrices are empty of rows.
               0.0 if one matrix is empty and the other is not.
    """
    n_rows_a, k_cols_a = matrix_a.shape
    n_rows_b, k_cols_b = matrix_b.shape

    if k_cols_a != k_cols_b:
        raise ValueError("Matrices must have the same number of columns (k).")

    # Handle edge cases with empty matrices
    if n_rows_a == 0 and n_rows_b == 0:
        return 1.0  # Two empty sets of rows can be considered perfectly similar
    if n_rows_a == 0 or n_rows_b == 0:
        return 0.0  # One empty and one non-empty matrix are not similar

    # 1. Calculate Jaccard Similarity for all pairs of rows
    # Similarity = 1 - Jaccard Distance
    # The similarity_matrix will have shape (n_rows_a, n_rows_b)
    similarity_matrix = np.zeros((n_rows_a, n_rows_b))

    for i in range(n_rows_a):
        for j in range(n_rows_b):
            row_a = matrix_a[i, :]
            row_b = matrix_b[j, :]
            # scipy.spatial.distance.jaccard computes the Jaccard *distance*
            # For boolean arrays, if both are all zeros, distance is 0 (similarity 1)
            # This is generally the desired behavior for identical all-zero patterns.
            # Ensure inputs are boolean for jaccard if they are int 0/1.
            # If they are already float 0.0/1.0, jaccard might treat them as continuous.
            # For binary (0/1) integer arrays, casting to bool is safest.
            try:
                j_distance = jaccard(row_a.astype(bool), row_b.astype(bool))
                similarity_matrix[i, j] = 1.0 - j_distance
            except ZeroDivisionError: 
                # This can happen if both rows are all zeros and the specific jaccard 
                # implementation doesn't gracefully handle 0/0.
                # Scipy's jaccard for boolean arrays should handle this (dist=0 if both all zero).
                # However, being explicit can be good.
                # If both rows sum to 0, they are identical in their emptiness.
                if np.sum(row_a) == 0 and np.sum(row_b) == 0:
                    similarity_matrix[i, j] = 1.0
                else:
                    similarity_matrix[i, j] = 0.0


    # 2. Construct the Cost Matrix for the assignment problem
    # The Hungarian algorithm (linear_sum_assignment) minimizes cost.
    # Cost = 1 - Similarity
    cost_matrix = 1.0 - similarity_matrix

    # 3. Solve the Assignment Problem
    # This finds the optimal pairing of rows from A to rows from B
    # (or vice-versa depending on which has fewer rows)
    # that minimizes the total cost.
    row_ind_a, col_ind_b = linear_sum_assignment(cost_matrix)

    # 4. Calculate the Overall Matrix Similarity
    # The number of matched pairs will be min(n_rows_a, n_rows_b)
    num_matched_pairs = len(row_ind_a)

    if num_matched_pairs == 0: # Should be covered by earlier checks, but as a safeguard
        return 0.0

    # Sum of similarities for the optimal matched pairs
    sum_optimal_similarities = similarity_matrix[row_ind_a, col_ind_b].sum()

    overall_similarity = sum_optimal_similarities / num_matched_pairs

    return overall_similarity

#go_feature_similarity = calculate_matrix_similarity(go_feature_matrix_dgd_shared.numpy(), go_feature_matrix_geneformer_shared.numpy())

In [17]:
go_feature_similarity12 = calculate_matrix_similarity(go_feature_matrix1_shared2.numpy(), go_feature_matrix2_shared1.numpy())
go_feature_similarity13 = calculate_matrix_similarity(go_feature_matrix1_shared3.numpy(), go_feature_matrix3_shared1.numpy())
go_feature_similarity23 = calculate_matrix_similarity(go_feature_matrix2_shared3.numpy(), go_feature_matrix3_shared2.numpy())

In [19]:
print(f"Similarity between SAE1 and SAE2: {go_feature_similarity12}")
print(f"Similarity between SAE1 and SAE3: {go_feature_similarity13}")
print(f"Similarity between SAE2 and SAE3: {go_feature_similarity23}")
print(f"Average similarity: {np.mean([go_feature_similarity12, go_feature_similarity13, go_feature_similarity23])}")

Similarity between SAE1 and SAE2: 1.0
Similarity between SAE1 and SAE3: 1.0
Similarity between SAE2 and SAE3: 1.0
Average similarity: 1.0
